# 📺 YouTube Trending Videos Analysis

A complete analysis of YouTube trending videos — covering top videos, dominant channels,
category distribution, and engagement patterns.

> **Running in demo mode?** This notebook uses realistic mock data by default.
> To switch to live data, add your YouTube Data API key in the **Configuration** cell below.

---

## 0 · Setup

In [ ]:
# Install dependencies if needed (uncomment)
# !pip install pandas matplotlib seaborn
# !pip install google-api-python-client   # only needed for live API data

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib
matplotlib.use('Agg')   # change to 'inline' if running in classic Jupyter
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from data_fetcher import generate_mock_data, save as save_data
from analyzer import (
    load, top_videos_by_views, top_channels, category_breakdown,
    engagement_stats, views_vs_engagement, channel_diversity
)
from visualizer import (
    chart_top_videos, chart_top_channels, chart_category_donut,
    chart_engagement_heatmap, chart_views_vs_likes, chart_summary_dashboard
)
from IPython.display import Image, display
print('✅ All imports OK')

## 1 · Configuration

In [ ]:
# ─── CONFIGURATION ─────────────────────────────────────────────────────────
API_KEY = None          # Paste your YouTube Data API v3 key here (or leave None for mock data)
REGION  = 'US'          # Region code: US, IN, GB, JP, BR, etc.
N       = 50            # Number of trending videos to analyse
# ───────────────────────────────────────────────────────────────────────────

from pathlib import Path
data_path = Path('data') / f'trending_{REGION}.json'

if API_KEY:
    from data_fetcher import fetch_live_data
    print(f'🌐 Fetching live data for region={REGION} ...')
    raw = fetch_live_data(API_KEY, REGION, N)
    save_data(raw, REGION)
elif data_path.exists():
    print(f'📂 Found existing data at {data_path}')
else:
    print(f'🎲 No API key provided — generating mock data (n={N}) ...')
    raw = generate_mock_data(N, REGION)
    save_data(raw, REGION)

df = load(str(data_path))
print(f'\n✅ Loaded {len(df)} videos.')
df.head()

## 2 · Engagement Summary

In [ ]:
stats = engagement_stats(df)

print('━' * 45)
print('  YouTube Trending — Engagement Summary')
print('━' * 45)
for k, v in stats.items():
    label = k.replace('_', ' ').title()
    val = f'{v:,}' if isinstance(v, int) else v
    print(f'  {label:<26} {val}')

## 3 · Summary Dashboard

In [ ]:
df_top = top_videos_by_views(df, 10)
df_cat = category_breakdown(df)

path = chart_summary_dashboard(stats, df_top, df_cat)
display(Image(str(path)))

## 4 · Top 10 Videos by Views

In [ ]:
print(df_top.to_string(index=False))
path = chart_top_videos(df_top)
display(Image(str(path)))

## 5 · Top Channels

In [ ]:
df_ch = top_channels(df, 10)
print(df_ch.to_string(index=False))

path = chart_top_channels(df_ch)
display(Image(str(path)))

## 6 · Category Breakdown

In [ ]:
print(df_cat.to_string(index=False))
path = chart_category_donut(df_cat)
display(Image(str(path)))

## 7 · Engagement Heatmap

In [ ]:
path = chart_engagement_heatmap(df_ch)
display(Image(str(path)))

## 8 · Views vs Like Rate

In [ ]:
df_eng = views_vs_engagement(df)
path = chart_views_vs_likes(df_eng)
display(Image(str(path)))

## 9 · Channel Diversity

In [ ]:
div = channel_diversity(df)
print('━' * 45)
for k, v in div.items():
    print(f'  {k.replace("_"," ").title():<30} {v}')

# What fraction of channels account for 50% of views?
ch_views = df.groupby('channel')['views'].sum().sort_values(ascending=False)
cumsum = ch_views.cumsum() / ch_views.sum()
n_50pct = (cumsum < 0.5).sum() + 1
print(f'\n  {n_50pct} channels account for 50% of total views.')

---
## 10 · Key Takeaways

Run the cell below to auto-generate insights from the data.

In [ ]:
top_cat   = df_cat.iloc[0]
top_chan  = df_ch.iloc[0]
top_vid   = df_top.iloc[0]

print('📌 Key Takeaways')
print('─' * 50)
print(f'• Most common category  : {top_cat["category"]} ({int(top_cat["video_count"])} videos)')
print(f'• Top channel           : {top_chan["channel"]} ({int(top_chan["appearances"])} trending videos)')
print(f'• Most-viewed video     : "{top_vid["title"][:55]}"')
print(f'  └─ Views              : {int(top_vid["views"]):,}')
print(f'• Avg like rate         : {stats["avg_like_rate"]}%')
print(f'• Channel diversity     : {div["unique_channels"]} unique channels in top {len(df)} trending videos')